##### DATA 620 - Project 2
## Large Two-Node (Bipartite) Network Analysis Using MovieLens
**Author**
- Barakat Adigun

## YOUTUBE RECORDING VIDEO LINK

https://youtu.be/pMHrPKBOXC8

# Project 2: Large Two-Node Network Analysis

## Introduction/Objective

The purpose of this project is to analyze a large two-node (bipartite) network consisting of two distinct groups of nodes. Unlike a traditional social network, edges exist only between the two groups and never within the same group.

For this project, I selected the MovieLens Latest Small dataset. The network consists of:

- Users
- Movies

An edge represents a user rating a movie.

To reduce the size of the network, I use an island method by retaining only strong ties (ratings of 4.0 or higher). I then analyze users and movies separately to avoid comparing different types of nodes directly.

In [121]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import requests
import zipfile
import io

## Step 1: Download and Load the Dataset

The MovieLens Latest Small dataset is downloaded directly from GroupLens.

The dataset contains ratings between users and movies.

Each row represents one interaction (edge) in the bipartite network.

In [122]:
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

response = requests.get(url)
zip_file = zipfile.ZipFile(io.BytesIO(response.content))

ratings = pd.read_csv(zip_file.open("ml-latest-small/ratings.csv"))
movies = pd.read_csv(zip_file.open("ml-latest-small/movies.csv"))

ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Step 2: Explore the Dataset

Before building the network, it is useful to understand the size of the dataset.

In [123]:
print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

ratings.describe()

Ratings: (100836, 4)
Movies: (9742, 3)


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


## Step 3: Reduce the Network (Island Method)

The original network contains over 100,000 ratings, making it difficult to interpret. To reduce the network, I retained only ratings of 4.0 or higher. This is similar to the island method described in the textbook because weaker ties are removed while stronger relationships remain. The resulting network emphasizes meaningful interactions between users and movies and is easier to analyze.

In [124]:
strong = ratings[ratings["rating"] >= 4.0]

print(strong.shape)

(48580, 4)


## Step 4: Build the Bipartite Network

The network consists of:

- User nodes
- Movie nodes

Edges exist only between users and movies.

There are no user-user edges or movie-movie edges.

In [125]:
B = nx.Graph()

user_nodes = []

movie_nodes = []

for row in strong.itertuples():

    user = f"user_{row.userId}"

    movie = f"movie_{row.movieId}"

    B.add_node(user, bipartite="User")

    B.add_node(movie, bipartite="Movie")

    B.add_edge(user, movie)

    user_nodes.append(user)

    movie_nodes.append(movie)

## Step 5: Examine the Network

The size of the reduced network is summarized below.

In [126]:
print("Bipartite graph created.")

print("Nodes:", B.number_of_nodes())

print("Edges:", B.number_of_edges())

Bipartite graph created.
Nodes: 6907
Edges: 48580


## Step 6: Keep the Largest Connected Component

Some nodes may become isolated after filtering.

Only the largest connected component is analyzed.

In [127]:
largest = max(nx.connected_components(B), key=len)

G = B.subgraph(largest).copy()

print(G.number_of_nodes())

print(G.number_of_edges())

6907
48580


## Step 7: Separate the Two Node Groups

This is the most important step.

Users and movies represent different entities.

Their centrality values should not be interpreted together because they answer different questions.

- High user degree = user rated many highly rated movies.

- High movie degree = movie received many high ratings.

In [128]:
users = [n for n,d in G.nodes(data=True) if d["bipartite"]=="User"]

movies_nodes = [n for n,d in G.nodes(data=True) if d["bipartite"]=="Movie"]

In [129]:
print("Number of users:", len(users))
print("Number of movies:", len(movies_nodes))

Number of users: 609
Number of movies: 6298


## Step 8: Calculate Degree Centrality Separately

Degree centrality is computed for the entire graph, but users and movies are interpreted separately.

In [130]:
degree = nx.degree_centrality(G)

print("Degree centrality calculated.")
print("Sample results:")
list(degree.items())[:10]

Degree centrality calculated.
Sample results:


[('user_1', 0.028960324355632783),
 ('movie_1', 0.021285838401390093),
 ('movie_3', 0.0026064291920069502),
 ('movie_6', 0.00999131190269331),
 ('movie_47', 0.020272227048942947),
 ('movie_50', 0.023602664349840718),
 ('movie_101', 0.0018824210831161307),
 ('movie_110', 0.02403706921517521),
 ('movie_151', 0.003185635679119606),
 ('movie_157', 0.0004344048653344917)]

In [131]:
user_degree = {k:v for k,v in degree.items() if k in users}

movie_degree = {k:v for k,v in degree.items() if k in movies_nodes}

In [132]:
print("Total degree centrality values:", len(degree))
print("Users analyzed:", len(user_degree))
print("Movies analyzed:", len(movie_degree))

Total degree centrality values: 6907
Users analyzed: 609
Movies analyzed: 6298


## Step 9: Most Connected Users

In [133]:
top_users = pd.DataFrame(

sorted(user_degree.items(),

key=lambda x:x[1],

reverse=True)[:10],

columns=["User","Degree Centrality"])

top_users

,User,Degree Centrality
0,user_414,0.177672
1,user_474,0.113959
2,user_380,0.096872
3,user_610,0.088908
4,user_606,0.088763
5,user_603,0.081523
6,user_105,0.080220
7,user_249,0.073994
8,user_182,0.070663
9,user_448,0.065885


### Interpretation

The users with the highest degree centrality are the most active users in the reduced network because they gave high ratings to many different movies. In this bipartite network, a high degree centrality value indicates that a user has many connections to movie nodes through strong ratings (4.0 or higher). Therefore, their high centrality reflects user activity rather than popularity.

## Step 10: Most Connected Movies

In [134]:
movie_lookup = dict(zip(movies.movieId,movies.title))

top_movies = pd.DataFrame(

sorted(movie_degree.items(),

key=lambda x:x[1],

reverse=True)[:10],

columns=["Movie","Degree Centrality"])

top_movies["MovieID"] = top_movies.Movie.str.replace("movie_","").astype(int)

top_movies["Title"] = top_movies.MovieID.map(movie_lookup)

top_movies

,Movie,Degree Centrality,MovieID,Title
0,movie_318,0.039676,318,"Shawshank Redemption, The (1994)"
1,movie_356,0.036056,356,Forrest Gump (1994)
2,movie_296,0.035332,296,Pulp Fiction (1994)
3,movie_593,0.032580,593,"Silence of the Lambs, The (1991)"
4,movie_2571,0.032146,2571,"Matrix, The (1999)"
5,movie_260,0.029105,260,Star Wars: Episode IV - A New Hope (1977)
6,movie_2959,0.025919,2959,Fight Club (1999)
7,movie_527,0.025340,527,Schindler's List (1993)
8,movie_1196,0.024327,1196,Star Wars: Episode V - The Empire Strikes Back...
9,movie_110,0.024037,110,Braveheart (1995)


### Interpretation

The movies with the highest degree centrality received high ratings from many different users. These movies occupy central positions within the movie group because they are connected to a large number of users through strong rating relationships. In this network, high degree centrality reflects how broadly connected a movie is within the filtered dataset rather than its average rating.

## Step 11: Visualize the Network

In [ ]:
plt.figure(figsize=(14,12))

pos = nx.spring_layout(G, seed=42)

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=users,
    node_color="skyblue",
    node_size=15,
    label="Users"
)

nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=movies_nodes,
    node_color="orange",
    node_size=15,
    label="Movies"
)

nx.draw_networkx_edges(
    G,
    pos,
    alpha=0.05,
    width=0.3
)

plt.title("Reduced Bipartite Network")
plt.legend()
plt.axis("off")
plt.show()

## Step 12: Discussion

The MovieLens dataset forms a bipartite network consisting of two distinct groups: users and movies. These groups are connected through rating relationships, with no edges between nodes of the same type. After applying an island method by retaining only ratings of 4.0 or higher, the network became smaller and easier to analyze while preserving the strongest connections.

From the user group, degree centrality identifies the most active users because they gave high ratings to many different movies. From the movie group, degree centrality identifies movies that are connected to many users through strong ratings, indicating that they are widely appreciated within the filtered network.

Users and movies represent different types of entities, so comparing their centrality values directly would not be meaningful. Instead, each group was analyzed separately, allowing degree centrality to measure importance within each node type. This approach provides a more accurate interpretation of the structure and relationships in a bipartite network.

# Conclusion

This project analyzed a large bipartite network using the MovieLens dataset. The network was reduced by applying an island method that retained only strong ties (ratings of 4.0 or higher). After reducing the network, users and movies were analyzed separately to avoid comparing different node types.

The results identified the most active users and the most connected movies based on degree centrality within their respective groups. Users with high degree centrality rated many highly rated movies, while movies with high degree centrality received high ratings from many different users. Separating the two node types avoids comparing different kinds of entities and provides a more meaningful interpretation of centrality in a bipartite network.